# MannuMitra — Machine Learning Experiment Module 03
## Random Forest + Histogram Gradient Boosting Hybrid Evaluation

**Objective:**
Evaluate whether combining Random Forest (RF) and Histogram Gradient Boosting (HGB) into a soft-voting or stacking ensemble outperforms the individual baseline models on the 30-crop Karnataka dataset.

**Architectures Compared:**
1. Random Forest Baseline (100 trees)
2. Histogram Gradient Boosting Baseline
3. Equal-Weight Soft-Voting Ensemble ($w_{RF}=0.5, w_{HGB}=0.5$)
4. Weighted Soft-Voting Ensemble (Weights selected via training-only 5-fold CV)
5. Stacking Ensemble (Base: RF + HGB, Meta-Classifier: Logistic Regression)

**Integrity Rules:**
- Strict 80/20 train/test split before ensembling.
- Untouched held-out test partition ($N=3,756$).
- Zero deployment of experimental models to production.


In [ ]:
import os
import json
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

ROOT_DIR = Path('..')
DATASET_PATH = ROOT_DIR / 'data' / 'rf_30_class' / 'karnataka_30_crops_cpcri_rainfall_fixed.csv'
if not DATASET_PATH.exists():
    DATASET_PATH = ROOT_DIR / 'data' / 'processed' / 'karnataka_30_crops_cpcri_rainfall_fixed.csv'
MODELS_DIR = ROOT_DIR / 'models' / 'rf_hgb_hybrid'

print('Dataset path:', DATASET_PATH)
print('Models directory:', MODELS_DIR)


### 1. Load Dataset & Train/Test Split (80/20)

In [ ]:
df = pd.read_csv(DATASET_PATH)
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
TARGET = 'label'

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df['label_enc'], test_size=0.20, random_state=42, stratify=df['label_enc']
)
print(f'Training samples: {len(X_train)}, Held-out test samples: {len(X_test)}')
print(f'Number of crops: {len(le.classes_)}')


### 2. Load Pre-Trained Hybrid Models or Train from Scratch
We inspect the pre-trained models saved in `models/rf_hgb_hybrid/`.

In [ ]:
models = {}
model_files = {
    'RF Baseline': MODELS_DIR / 'rf_baseline.pkl',
    'HGB Baseline': MODELS_DIR / 'hgb_baseline.pkl',
    'Equal Soft Voting': MODELS_DIR / 'hybrid_equal_soft_voting.pkl',
    'Weighted Soft Voting': MODELS_DIR / 'hybrid_weighted_soft_voting.pkl',
    'Stacking Classifier': MODELS_DIR / 'hybrid_stacking_classifier.pkl'
}

for name, path in model_files.items():
    if path.exists():
        models[name] = joblib.load(path)
        print(f'Loaded {name} from {path}')
    else:
        print(f'{path} not found. Training {name} on the fly...')
        if name == 'RF Baseline':
            m = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
        elif name == 'HGB Baseline':
            m = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(X_train, y_train)
        elif name == 'Equal Soft Voting':
            rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            hgb = HistGradientBoostingClassifier(max_iter=100, random_state=42)
            m = VotingClassifier([('rf', rf), ('hgb', hgb)], voting='soft').fit(X_train, y_train)
        elif name == 'Weighted Soft Voting':
            rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            hgb = HistGradientBoostingClassifier(max_iter=100, random_state=42)
            m = VotingClassifier([('rf', rf), ('hgb', hgb)], voting='soft', weights=[0.4, 0.6]).fit(X_train, y_train)
        elif name == 'Stacking Classifier':
            rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            hgb = HistGradientBoostingClassifier(max_iter=100, random_state=42)
            m = StackingClassifier([('rf', rf), ('hgb', hgb)], final_estimator=LogisticRegression(max_iter=500), cv=5).fit(X_train, y_train)
        models[name] = m


### 3. Evaluate Models on Held-Out Test Set
Evaluate test accuracy, macro F1, weighted F1, and balanced accuracy.

In [ ]:
results = []
for name, m in models.items():
    preds = m.predict(X_test)
    acc = accuracy_score(y_test, preds) * 100
    macro_f1 = f1_score(y_test, preds, average='macro') * 100
    weighted_f1 = f1_score(y_test, preds, average='weighted') * 100
    bal_acc = balanced_accuracy_score(y_test, preds) * 100
    results.append({
        'Model': name,
        'Test Accuracy (%)': round(acc, 2),
        'Macro F1 (%)': round(macro_f1, 2),
        'Weighted F1 (%)': round(weighted_f1, 2),
        'Balanced Accuracy (%)': round(bal_acc, 2)
    })

df_res = pd.DataFrame(results)
df_res


### 4. Key Agronomic Insights & Focus Pair Error Analysis
Inspect whether ensembling resolves confusion between Mungbean/Blackgram and Arecanut/Coconut.

In [ ]:
pairs = [('mungbean', 'blackgram'), ('arecanut', 'coconut')]
for name, m in models.items():
    preds = m.predict(X_test)
    print(f'=== {name} ===')
    for c1, c2 in pairs:
        enc1, enc2 = le.transform([c1])[0], le.transform([c2])[0]
        mask_c1 = (y_test == enc1)
        c1_corr = (preds[mask_c1] == enc1).sum()
        c1_to_c2 = (preds[mask_c1] == enc2).sum()
        print(f'  {c1} (N={mask_c1.sum()}): {c1_corr} correct, {c1_to_c2} misclassified as {c2}')
